# Lecture 12 notebook: the tape

This notebook replays Lecture 12 for any USD-M perpetual on Binance and any period: it downloads the aggregated trades, checks them,
finds and patches holes, builds the derived tables, and then explores and describes the data. Everything runs on an ordinary computer:
the pipeline works one month, and within a month one day, at a time.

| § | Content | Lecture |
|---|---|---|
| 0 | Settings: market, period, data folder; what to expect in time and memory | Appendix A.2 |
| 1 | The pipeline: download and check, find holes, patch, clean, aggregate, sequence | §4, §5 |

The pipeline code is in `tape.py`, next to this notebook. Data are written to `data/` at the top of the repository, which git ignores:
nothing downloaded is ever committed. To keep data elsewhere, set the environment variable `TAPE_ROOT` before starting Jupyter.

## 0. Settings

Choose the market and the period (months, `YYYY-MM`). The default, August and September 2022, contains the archive's holes of §5.2:
four missing days in August and September, a day missing on 10 September, two duplicated days, and the six minutes missing from the archive on 6 September.

In [ ]:
import os, sys, time
os.environ.setdefault("POLARS_MAX_THREADS", "4")      # fewer threads: lower peak memory, longer run
import polars as pl, psutil
import tape

SYMBOL = "BTCUSDT"     # any USD-M perpetual, e.g. "ETHUSDT"
START = "2022-08"      # first month
END = "2022-09"        # last month

pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(60)
print("data folder:", tape.ROOT)
print(f"free memory: {psutil.virtual_memory().available / 1e9:.1f} GB of {psutil.virtual_memory().total / 1e9:.1f} GB; "
      f"each stage waits until {tape.MIN_FREE_GB} GB are free")

**What to expect.** Measured on a computer with 16 GB of memory and four threads (Appendix A.2), per month:

| Stage | Peak memory | Time per month |
|---|---|---|
| Get: download, convert, check | about 2.3 GB | 1–2 minutes, mostly downloading |
| Find holes, patch, clean | about 3.2 GB | a few minutes for the whole period |
| Aggregate | 1.9–3.1 GB | 5 to 15 seconds |
| Sequence | 1.6–3.2 GB | 30 seconds to 2 minutes; longer in busy months |

A month of BTCUSDT takes about 0.4 GB of disk as Parquet. Months already downloaded are not downloaded again,
so the cell below can be rerun after an interruption.

## 1. The pipeline

`tape.run_pipeline` runs the six stages of §4.3, each stage and month in a fresh process, so that its memory is freed and its peak measured:

1. **get**: download the monthly zip, verify its checksum, convert to Parquet, check rows, IDs and time order, log, delete the zip;
2. **find holes**: every jump and every repeat in the trade IDs, across month boundaries;
3. **patch**: for each missing range, download the daily files of the days it spans, and check that they fill it exactly;
4. **clean**: for months with holes or repeats only, write raw + patches − duplicates;
5. **aggregate**: counts, volumes, histograms and quantiles, with Polars' streaming engine;
6. **sequence**: day by day, with the last trades of each day carried into the next: bars, realised variance, durations, persistence, flags.

In [ ]:
t0 = time.time()
holes, patches, log = tape.run_pipeline(SYMBOL, START, END)
print(f"done in {(time.time() - t0) / 60:.1f} minutes")

**Time and peak memory**, per stage and month: `waited_s` is the time spent waiting for free memory, `run_s` the work itself; months already downloaded are not fetched again.

In [ ]:
log

**The holes.** Each row is a jump in the trade IDs (missing trades) or a day on which IDs repeat (duplicated rows).
Times are those of the last trade before and the first trade after the hole.

In [ ]:
def show_holes(h):
    return h.with_columns(pl.from_epoch("from_ms", time_unit="ms").alias("from"), pl.from_epoch("to_ms", time_unit="ms").alias("to"),
                          ((pl.col("to_ms") - pl.col("from_ms")) / 60_000).round(1).alias("minutes")).select("kind", "month", "n", "from", "to", "minutes")
show_holes(holes)

**The patches.** For every missing range: how many of the missing trades the daily files contain, and the verdict.
"Filled" means the patch closes the hole exactly; "not in the archive" means Binance's daily files miss the same trades.

In [ ]:
show_holes(patches).with_columns(patches["patched"], patches["status"]) if patches.height else "no missing trades in this period"

In [ ]:
filled = patches.filter(pl.col("status") == "filled")["n"].sum() if patches.height else 0
missing = holes.filter(pl.col("kind") == "missing")["n"].sum()
print(f"missing trades: {missing:,}; filled exactly from the daily files: {filled:,}; left unfilled and recorded: {missing - filled:,}")